# 🤖 What is an AI Agent?

An **AI Agent** is essentially a function you write to perform specific tasks that an LLM cannot do on its own.

## Why Do We Need AI Agents?

LLMs accessed via API have limitations:

* **Calculations**: When you ask for a calculation, the LLM will only *predict* the answer based on patterns, not perform actual computation. This can lead to inaccurate results.
* **Real-time Data**: LLMs cannot access live information like:
  * Current weather in a city
  * Live cryptocurrency prices
  * Stock market data
  * Database queries
  * File system operations

## How AI Agents Work

You create functions in your Python code that:
1. Get executed when the user asks specific types of questions
2. Perform the actual computation or data retrieval
3. Return accurate, real-time results to the LLM
4. Allow the LLM to incorporate this data into its response

## Terminology

These functions are known by several names:
* **AI Agents**
* **Tools**
* **Functions**
* **Function Calling**

## Example Use Cases

| User Request | AI Agent Function |
|--------------|-------------------|
| "Calculate 2847 × 3921" | `calculator(expression)` |
| "What's the weather in New York?" | `get_weather(city)` |
| "What's the current Bitcoin price?" | `get_crypto_price(symbol)` |
| "Search my emails for invoices" | `search_emails(query)` |
| "Query our sales database" | `execute_sql_query(sql)` |

![image_1776939199413.png](./image_1776939199413.png "AI Agent Architecture Diagram")

## Key Takeaway

AI Agents bridge the gap between what LLMs can predict (text generation) and what they cannot do (computation, real-time data access, external system integration).






In [0]:
%pip install groq

import sys
import json
from groq import Groq
import os

# Add parent folder (one level above GenAI-AgenticAI) to Python path to import config
parent_dir = os.path.dirname(os.path.dirname(os.path.abspath('')))
sys.path.insert(0, parent_dir)

# Import API key from config file (stored outside git repo)
from groq_config import GROQ_API_KEY

# Initialize Groq client
client = Groq(api_key=GROQ_API_KEY)

# ═══════════════════════════════════════════════════════════════
# Step 1: Define the AI Agent (Tool/Function)
# ═══════════════════════════════════════════════════════════════

def calculate(expression: str) -> float:
    """AI Agent that performs mathematical calculations."""
    try:
        # Safely evaluate the mathematical expression
        result = eval(expression, {"__builtins__": {}}, {})
        return float(result)
    except Exception as e:
        return f"Error: {str(e)}"

# ═══════════════════════════════════════════════════════════════
# Step 2: Define Tool Schema for the LLM
# ═══════════════════════════════════════════════════════════════

tools = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Performs mathematical calculations. Use this when the user asks for any calculation.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "The mathematical expression to evaluate (e.g., '2847 * 3921')"
                    }
                },
                "required": ["expression"]
            }
        }
    }
]

# ═══════════════════════════════════════════════════════════════
# Step 3: Ask the LLM a Question That Requires Tool Use
# ═══════════════════════════════════════════════════════════════

user_question = "What is 2847 multiplied by 3921?"

print("="*60)
print(f"🧑 USER: {user_question}")
print("="*60)

# Call the LLM with the tools available
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": user_question}],
    tools=tools,
    tool_choice="auto"  # Let the model decide if/when to use tools
)

message = response.choices[0].message
print(message)
# ═══════════════════════════════════════════════════════════════
# Step 4: Check if the LLM Wants to Use a Tool
# ═══════════════════════════════════════════════════════════════

if message.tool_calls:
    print("\n🤖 LLM DECISION: I need to use a tool to calculate this!\n")
    
    # Extract the tool call details
    tool_call = message.tool_calls[0]
    function_name = tool_call.function.name
    function_args = json.loads(tool_call.function.arguments)
    
    print(f"📞 TOOL CALL:")
    print(f"   Function: {function_name}")
    print(f"   Arguments: {function_args}")
    
    # ═══════════════════════════════════════════════════════════════
    # Step 5: Execute the Tool (AI Agent)
    # ═══════════════════════════════════════════════════════════════
    
    if function_name == "calculate":
        result = calculate(function_args["expression"])
        print(f"\n⚙️  TOOL EXECUTION: calculate('{function_args['expression']}')")
        print(f"   Result: {result}")
    
    # ═══════════════════════════════════════════════════════════════
    # Step 6: Send Tool Result Back to LLM for Final Response
    # ═══════════════════════════════════════════════════════════════
    
    # Prepare messages with tool result
    messages = [
        {"role": "user", "content": user_question},
        message,  # The assistant's decision to use the tool
        {
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": str(result)
        }
    ]
    
    # Get final response from LLM
    final_response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages
    )
    
    print(f"\n🤖 LLM FINAL RESPONSE: {final_response.choices[0].message.content}")
    
else:
    # LLM responded directly without using tools
    print(f"\n🤖 LLM RESPONSE (no tool used): {message.content}")

print("\n" + "="*60)
print("✅ AI Agent Demo Complete!")
print("="*60)

In [27]:
%pip install duckduckgo

from groq import Groq
import os
import sys
import json

# Add parent folder (one level above GenAI-AgenticAI) to Python path to import config
parent_dir = os.path.dirname(os.path.dirname(os.path.abspath('')))
sys.path.insert(0, parent_dir)

# Import API key from config file (stored outside git repo)
from groq_config import GROQ_API_KEY

# Initialize Groq client
client = Groq(api_key=GROQ_API_KEY)

# ═══════════════════════════════════════════════════════════════
# Step 1: Define the AI Agent (Tool/Function) - Web Search
# ═══════════════════════════════════════════════════════════════

from ddgs import DDGS

def web_search(query: str) -> str:
    """AI Agent that searches the web for current information."""
    try:
        ddgs = DDGS()
        # Add "current" to the query for better weather results
        search_query = query if "weather" in query.lower() else query
        results = ddgs.text(search_query, max_results=5)
        
        if not results:
            return "No results found."
        
        # Format results as a string
        formatted = []
        for r in results:
            formatted.append(f"- {r['title']}: {r['body']}")
        
        return "\n".join(formatted)
    except Exception as e:
        return f"Error: {str(e)}"

# ═══════════════════════════════════════════════════════════════
# Step 2: Define Tool Schema for the LLM
# ═══════════════════════════════════════════════════════════════

tools = [
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Performs a web search to get current information. Use when user asks about weather, time, news, or current events.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query to find information"
                    }
                },
                "required": ["query"]
            }
        }
    }
]

# ═══════════════════════════════════════════════════════════════
# Step 3: Ask the LLM a Question That Requires Web Search
# ═══════════════════════════════════════════════════════════════

user_question = "What's the current weather in Hyderabad?"

print("="*60)
print(f"USER: {user_question}")
print("="*60)

# Call the LLM with the tools available
response = client.chat.completions.create(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    messages=[{"role": "user", "content": user_question}],
    tools=tools,
    tool_choice="auto"
)

message = response.choices[0].message

# ═══════════════════════════════════════════════════════════════
# Step 4: Check if the LLM Wants to Use a Tool
# ═══════════════════════════════════════════════════════════════

if message.tool_calls:
    print("\nLLM wants to use a tool!\n")
    
    # Extract the tool call details
    tool_call = message.tool_calls[0]
    function_name = tool_call.function.name
    function_args = json.loads(tool_call.function.arguments)
    
    print(f"Tool: {function_name}")
    print(f"Args: {function_args}")
    
    # ═══════════════════════════════════════════════════════════════
    # Step 5: Execute the Tool (AI Agent)
    # ═══════════════════════════════════════════════════════════════
    
    if function_name == "web_search":
        result = web_search(function_args["query"])
        print(f"\nSearch result: {result[:200]}...")
    
    # ═══════════════════════════════════════════════════════════════
    # Step 6: Send Tool Result Back to LLM for Final Response
    # ═══════════════════════════════════════════════════════════════
    
    messages = [
        {"role": "user", "content": user_question},
        {"role": "assistant", "tool_calls": message.tool_calls},
        {
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": str(result)
        }
    ]
    
    # Get final response from LLM
    final_response = client.chat.completions.create(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        messages=messages
    )
    
    print(f"\nLLM FINAL: {final_response.choices[0].message.content}")
    
else:
    print(f"\nLLM Response: {message.content}")

print("\n" + "="*60)
print("Web Search Agent Demo Complete!")
print("="*60)


  Using cached duckduckgo-0.1.tar.gz (2.9 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [22 lines of output]
      Traceback (most recent call last):
        File "c:\Program Files\Python311\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 353, in <module>
          main()
        File "c:\Program Files\Python311\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 335, in main
          json_out['return_val'] = hook(**hook_input['kwargs'])
                                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        File "c:\Program Files\Python311\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 118, in get_requires_for_build_wheel
          return hook(config_settings)
                 ^^^^^^^^^^^^^^^^^^^^^
        File "C:\Users\vinit\AppData\Local\Temp\pip-build-env-vve3af4z\overlay\Lib\site-packages\setuptools\build_meta.py", line 333, in get_requires_for_b

USER: What's the current weather in Hyderabad?

LLM wants to use a tool!

Tool: web_search
Args: {'query': 'Hyderabad current weather'}

Search result: - Current weather and temperature in Hyderabad: Hourly and weekly: Current temperature ... Today's weather will see a mix of conditions throughout the day. ... Hyderabad
- Hyderabad Weather. 14 Day Fo...

LLM FINAL: The current weather in Hyderabad is not explicitly stated in the search results. However, I can suggest some ways to find the current weather:

1. Check online weather websites such as WorldWeatherOnline.com or AccuWeather.com for the latest updates.
2. Look for weather apps on your smartphone, such as Dark Sky or Weather Underground, which provide real-time weather conditions.
3. Tune into local news channels or radio stations for weather updates.

I hope this helps! Let me know if you have any other questions.

Web Search Agent Demo Complete!


In [1]:
%pip install groq requests

from groq import Groq
import os
import sys
import json
import requests
from datetime import datetime, timedelta


# Add parent folder (one level above GenAI-AgenticAI) to Python path to import config
parent_dir = os.path.dirname(os.path.dirname(os.path.abspath('')))  
sys.path.insert(0, parent_dir)

from groq_config import GROQ_API_KEY

# Initialize Groq client
client = Groq(api_key=GROQ_API_KEY)



#sum of 2 numbers 

def sum_of_numbers(num1: float, num2: float) -> float:
    """AI Agent that calculates the sum of two numbers."""
    return num1 + num2

def calucator(expression: str) -> float:
    """AI Agent that performs mathematical calculations."""
    try:
        # Safely evaluate the mathematical expression   
        result = eval(expression, {"__builtins__": {}}, {})
        return float(result)
    except Exception as e:
        return f"Error: {str(e)}"

# Weather API Configuration
WEATHER_API_KEY = "a0cdb3ab4cb143e5ae994433262504"
WEATHER_BASE_URL = "http://api.weatherapi.com/v1"

def weather_update(city: str, date_input: str = "today") -> str:
    """
    Get weather update for a city with flexible date input.
    
    Args:
        city: Name of the city
        date_input: Can be:
            - "today" or "current" - today's weather
            - "tomorrow" - tomorrow's weather
            - "1", "2", "3", etc. - days ahead (as string or int)
            - "in 2 days", "in 3 days" - relative days ahead
            - "2026-04-27" - specific date (YYYY-MM-DD format)
    
    Returns:
        Weather information as a string
    """
    try:
        # Parse the date input
        date_input_lower = str(date_input).lower().strip()
        today = datetime.now()
        
        # Determine days ahead based on input
        if date_input_lower in ["today", "current", "0"]:
            days = 0
        elif date_input_lower in ["tomorrow", "1"]:
            days = 1
        elif "day" in date_input_lower:
            # Handle "in 2 days", "2 days", etc.
            import re
            match = re.search(r'(\d+)', date_input_lower)
            if match:
                days = int(match.group(1))
            else:
                return "Error: Could not parse date. Use 'today', 'tomorrow', or 'in X days'"
        else:
            # Try parsing as a specific date
            try:
                target_date = datetime.strptime(str(date_input), "%Y-%m-%d")
                days = (target_date - today).days
                if days < 0:
                    return "Error: Cannot get weather for past dates"
                if days > 14:
                    return "Error: Can only forecast up to 14 days ahead"
            except ValueError:
                return f"Error: Invalid date format. Use 'today', 'tomorrow', 'in 2 days', or 'YYYY-MM-DD'"
        
        # Determine the endpoint based on days
        if days == 0:
            # Current weather
            endpoint = f"{WEATHER_BASE_URL}/current.json"
            params = {"key": WEATHER_API_KEY, "q": city, "aqi": "no"}
        else:
            # Forecast weather
            endpoint = f"{WEATHER_BASE_URL}/forecast.json"
            params = {"key": WEATHER_API_KEY, "q": city, "days": days + 1, "aqi": "no"}
        
        # Make the API request
        response = requests.get(endpoint, params=params)
        response.raise_for_status()
        
        data = response.json()
        
        # Format the response
        if days == 0:
            location = data["location"]
            current = data["current"]
            result = f"Weather in {location['name']}, {location['country']}:\n"
            result += f"Temperature: {current['temp_c']}°C ({current['temp_f']}°F)\n"
            result += f"Condition: {current['condition']['text']}\n"
            result += f"Humidity: {current['humidity']}%\n"
            result += f"Wind: {current['wind_kph']} km/h\n"
            result += f"Feels like: {current['feelslike_c']}°C"
        else:
            location = data["location"]
            forecast = data["forecast"]["forecastday"][days]
            result = f"Weather forecast for {location['name']}, {location['country']}:\n"
            result += f"Date: {forecast['date']}\n"
            result += f"High: {forecast['day']['maxtemp_c']}°C\n"
            result += f"Low: {forecast['day']['mintemp_c']}°C\n"
            result += f"Condition: {forecast['day']['condition']['text']}\n"
            result += f"Rain chance: {forecast['day']['daily_chance_of_rain']}%"
        
        return result
        
    except requests.exceptions.HTTPError as e:
        return f"Error: {e.response.status_code} - {e.response.json().get('error', {}).get('message', 'Unknown error')}"
    except Exception as e:
        return f"Error: {str(e)}"

tools = [
            {
                "type": "function",
                "function": {
                    "name": "weather_update",
                    "description": "Get weather update for a city with flexible date input.",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "city": {
                                "type": "string",
                                "description": "Name of the city"
                            },
                            "date_input": {
                                "type": "string",
                                "description": "Date for weather update. Can be 'today', 'tomorrow', 'in 2 days', or a specific date like '2026-04-27'."
                            }
                        },
                        "required": ["city", "date_input"]
                    }
                }
            },
            {
                "type": "function",
                "function": {
                    "name": "calucator",
                    "description": "Performs mathematical calculations.",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "expression": {
                                "type": "string",
                                "description": "The mathematical expression to evaluate (e.g., '2847 * 3921')"
                            }
                        },
                        "required": ["expression"]
                    }
                }
            },
            {
                "type": "function",
                "function": {
                    "name": "sum_of_numbers",
                    "description": "Calculates the sum of two numbers.",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "num1": {
                                "type": "number",
                                "description": "The first number"
                            },
                            "num2": {
                                "type": "number",
                                "description": "The second number"
                            }
                        },
                        "required": ["num1", "num2"]
                    }
                }
            }
        ]
def main(user_question: str):
    """Main function to handle user questions with AI agents."""
    # Initialize messages for this question
    messages = [{"role": "user", "content": user_question}]
    
    # First call to LLM
    response = client.chat.completions.create(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        messages=messages,
        tools= tools,
        tool_choice="auto",
    )
    
    message = response.choices[0].message
    
    # Check if LLM wants to use a tool
    while message.tool_calls:
        tool_call = message.tool_calls[0]
        print(f"LLM wants to use tool: {tool_call.function.name} with arguments: {tool_call.function.arguments}")
        
        function_name = tool_call.function.name
        function_args = json.loads(tool_call.function.arguments)
        
        # Execute the appropriate function
        if function_name == "weather_update":
            result = weather_update(function_args["city"], function_args["date_input"])
            print(f"Weather Update Result:\n{result}")
        elif function_name == "calucator":
            result = calucator(function_args["expression"])
            print(f"Calculator Result: {result}")
        elif function_name == "sum_of_numbers":
            result = sum_of_numbers(function_args["num1"], function_args["num2"])
            print(f"Sum of Numbers Result: {result}")
        
        # Add tool result to messages
        messages.append({
            "role": "assistant",
            "tool_calls": message.tool_calls
        })
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": str(result)
        })
        
        # Get next response from LLM
        response = client.chat.completions.create(
            model="meta-llama/llama-4-scout-17b-16e-instruct",
            messages=messages
        )
        
        message = response.choices[0].message
    
    # No tool calls - return final response
    return message.content

user_question = input("Ask a question that may require tool use (e.g., weather update, calculation): ")
main_response = main(user_question)
    
print(f"\nFinal LLM Response:\n{main_response}")



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.

Final LLM Response:
Got it.
